# code-server on Colab

Run all. No clone, no credentials needed — with no key it falls back to a
Cloudflare quick tunnel. Add `TS_AUTHKEY` for a **fixed** `https://<node>.<tailnet>.ts.net`.

> Colab's FAQ does not permit remote-IDE use and Colab warns when it detects one.
> Fine for a host you control; don't lean on it here.

## 1. Scripts

In [ ]:
import pathlib, urllib.request

REF, REPO = "main", "recursive-factory/code-server"   # pin a commit SHA to bust the CDN cache
BASE = pathlib.Path("/content/cs")

for rel in ["lib/common.sh", "install.sh", "serve.sh", "supervise.sh", "stop.sh", "doctor.sh"]:
    f = BASE / rel
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_text(urllib.request.urlopen(
        f"https://raw.githubusercontent.com/{REPO}/{REF}/colab/{rel}", timeout=30).read().decode())
    if f.suffix == ".sh":
        f.chmod(0o755)
print("fetched ->", BASE)

## 2. Configure

In [ ]:
import os

# Leave empty to use a real environment variable, or a Colab Secret of that name.
NGROK_TOKEN = ""
TS_AUTHKEY  = ""      # tskey-auth-... reusable key => fixed https://code.<tailnet>.ts.net

TUNNEL        = ""    # "" = auto | ngrok | cloudflared | tailscale | none
TS_HOSTNAME   = ""    # "" = "code"
NGROK_DOMAIN  = ""    # reserved ngrok domain => stable URL
CS_PORT       = ""    # "" = first free port in CS_PORT_RANGE
CS_PORT_RANGE = ""    # "" = 3000-5000
IDLE          = 1800  # stop after 30 min with no connections (0 disables)
MAX_RUNTIME   = 39600 # hard stop at 11 h

def _secret(name):
    """Colab Secrets are not environment variables; userdata.get is the only way in."""
    try:
        from google.colab import userdata
    except Exception:
        return ""
    try:
        return (userdata.get(name) or "").strip()
    except Exception as e:
        if "access" in str(e).lower():
            print(f"[secret] grant notebook access for '{name}' (key icon)")
        return ""

def _pick(cell_value, name):
    return cell_value.strip() or os.environ.get(name, "").strip() or _secret(name)

tok, tskey = _pick(NGROK_TOKEN, "NGROK_TOKEN"), _pick(TS_AUTHKEY, "TS_AUTHKEY")

if tok:   os.environ["NGROK_TOKEN"] = tok
if tskey: os.environ["TS_AUTHKEY"] = tskey
os.environ.update(CS_TUNNEL=TUNNEL, CS_IDLE_TIMEOUT=str(IDLE),
                  CS_MAX_RUNTIME=str(MAX_RUNTIME), CS_PORT=str(CS_PORT))
for key, value in (("CS_PORT_RANGE", CS_PORT_RANGE), ("NGROK_DOMAIN", NGROK_DOMAIN),
                   ("TS_HOSTNAME", TS_HOSTNAME)):
    if value:
        os.environ[key] = value

mask = lambda s: f"{s[:13]}…({len(s)})" if s else "none"
print("tunnel:", TUNNEL or "auto", "| port:", CS_PORT or f"auto ({CS_PORT_RANGE or '3000-5000'})")
print("ngrok:", mask(tok), "| tailscale:", mask(tskey), "| node:", TS_HOSTNAME or "code")

## 3. Install

In [ ]:
import os, subprocess

# Only fetch tailscale when it will actually be used (auto picks it if a key exists).
want_ts = "1" if (TUNNEL == "tailscale" or os.environ.get("TS_AUTHKEY")) else "0"
subprocess.run(["bash", f"{BASE}/install.sh"], check=True,
               env={**os.environ, "WITH_CLOUDFLARED": "1", "WITH_TAILSCALE": want_ts})

## 4. Start (detached, returns immediately)

In [ ]:
import os, pathlib, subprocess

STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
STATE.mkdir(parents=True, exist_ok=True)

# stop.sh retires any previous supervisor and reclaims the port, so this is re-runnable.
subprocess.run(["bash", f"{BASE}/stop.sh"], check=False,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
(STATE / "credentials.txt").unlink(missing_ok=True)

with open(STATE / "supervise.out", "w") as fh:
    proc = subprocess.Popen(["bash", f"{BASE}/supervise.sh"], stdout=fh,
                            stderr=subprocess.STDOUT, start_new_session=True)
(STATE / "supervise.pid").write_text(str(proc.pid))
print("supervisor", proc.pid, "| log:", STATE / "supervise.out")

## 5. URL

Waits at most 30 s. The stack keeps running in the background, so re-run this
cell instead of blocking the kernel.

In [ ]:
import os, time, pathlib, subprocess
from IPython.display import display, clear_output, HTML

STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
creds, log = STATE / "credentials.txt", STATE / "supervise.out"

for _ in range(30):
    info = {}
    if creds.exists():
        for line in creds.read_text().splitlines():
            if "=" in line:
                key, value = line.split("=", 1)
                info[key] = value
    clear_output(wait=True)
    if info.get("url"):
        display(HTML(f'<p style="font-size:17px"><b>code-server</b> '
                     f'<a href="{info["url"]}" target="_blank">{info["url"]}</a></p>'
                     f'<p><b>password</b> <code>{info.get("password")}</code></p>'))
        print("tunnel:", info.get("tunnel"), "| port:", info.get("port"))
        break
    print("\n".join(log.read_text().splitlines()[-6:]) if log.exists() else "(starting)")
    time.sleep(1)
else:
    print("not up yet — re-run this cell. It runs in the background.\nlog:", log)
    print(subprocess.run(["bash", f"{BASE}/doctor.sh"], capture_output=True,
                         text=True).stdout[-800:])

## 6. Stop (opt-in)

In [ ]:
STOP = False        # set True, then run THIS cell alone: "Run all" executes it too

import os, pathlib, signal, subprocess

STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
if not STOP:
    print("skipped — stopping is opt-in so Run all cannot kill the stack.")
else:
    pid = STATE / "supervise.pid"
    if pid.exists():
        try:
            os.killpg(os.getpgid(int(pid.read_text())), signal.SIGTERM)
        except (ValueError, ProcessLookupError):
            pass
        pid.unlink(missing_ok=True)
    subprocess.run(["bash", f"{BASE}/stop.sh"], check=False)

## Troubleshooting

| symptom | cause |
|---|---|
| `Funnel is disabled for this tailnet` | approve the printed `login.tailscale.com/f/funnel?node=…` link once |
| ts.net hostname resets TLS / times out | same — ingress refuses until Funnel is enabled, even though `funnel status` looks fine |
| blank page | quick-tunnel interstitial not clicked through (ngrok/cloudflared) |
| `safesocket.Listen: address already in use` | fixed: run cell 1 to refresh the scripts |
| URL died seconds after appearing | `Run all` ran the Stop cell — it is opt-in now |

Scripts live in the repo at `colab/`; cell 1 re-fetches them.